# 01 USAspending pilot: PIDP, BRIC and EDA in coastal CA/OR/WA

**Task:** T1.1 (Coastal Harbor Funding Equity)

**Reads:** `data/processed/usaspending_pilot/{prime_awards,prime_transactions,subawards}.csv` and `process_log.json`, written by `uv run python -m harborfund.ingest.usaspending all`. Also reads `data/reference/coastal_counties.csv`.

**Writes:** nothing. The outputs are the pilot report, so they are kept when committed. They hold aggregates only, no raw records.

**Scope used here:**
- PIDP and EDA: an award counts if its recipient county or place-of-performance county is one of the 42 NOAA coastal shoreline counties.
- BRIC (97.047) prime awards: every prime in CA/OR/WA counts, since the recipient is usually the state. BRIC subawards are county-flagged from ZIP codes.
- Window FY2010–FY2026 (BRIC FY2020+). Award-level tables use the base action FY. Dollar tables by FY use transaction obligations by action FY.
- Dollars are nominal.

In [ ]:
import json
import numpy as np
import pandas as pd
from harborfund import config

OUT = config.PROCESSED_DIR / "usaspending_pilot"
log = json.loads((OUT / "process_log.json").read_text())
BOOL = ["in_fy_window", "has_uei", "has_duns", "recipient_is_state_gov", "prime_recipient_is_state_gov",
        "state_level_prime", "recipient_in_coastal_county", "pop_in_coastal_county", "in_coastal_county",
        "in_states", "earmark_text_flag", "earmark_candidate_year", "earmark_flag", "eda_rlf_flag",
        "harbor_kw_flag", "in_scope", "zip_multi_county", "zip_unmatched"]

def load(name):
    df = pd.read_csv(OUT / f"{name}.csv", dtype=str, low_memory=False)
    for c in BOOL:
        if c in df:
            df[c] = df[c].map({"True": True, "False": False}).fillna(False).astype(bool)
    df["fy"] = pd.to_numeric(df["fy"], errors="coerce").astype("Int64")
    for c in [c for c in df if c.endswith("_nominal")]:
        df[c] = pd.to_numeric(df[c], errors="coerce")
    for c in ("recipient_state", "pop_state", "geo_match_basis"):
        df[c] = df[c].fillna("")
    # State the row is counted in: the matched coastal county's state; otherwise
    # (state-level BRIC primes) the recipient state if in CA/OR/WA, else the POP state.
    rec_ok = df["recipient_state"].isin(["CA", "OR", "WA"])
    df["geo_state"] = np.select(
        [df["recipient_in_coastal_county"], df["pop_in_coastal_county"], rec_ok],
        [df["recipient_state"], df["pop_state"], df["recipient_state"]], default=df["pop_state"])
    return df

awards, txns, subs = (load(n) for n in ("prime_awards", "prime_transactions", "subawards"))
print("raw run:", log["raw_run"], "| processed:", log["processed_utc"])
pd.DataFrame({k: v["raw_rows"] | {"kept_" + kk: vv for kk, vv in v["in_scope_rows"].items()}
              for k, v in log["downloads"].items()}).T

## 1. Scope and award types

In [ ]:
A = awards[awards.in_scope & awards.in_fy_window].copy()
T = txns[txns.in_scope & txns.in_fy_window].copy()
S = subs[subs.in_fy_window].copy()
print(f"prime awards in scope: {len(A):,} | transactions: {len(T):,} | subawards in CA/OR/WA: {len(S):,} "
      f"(in coastal counties: {S.in_coastal_county.sum():,})")
pd.crosstab(A["program_id"], A["assistance_type_description"].fillna("(blank)"), margins=True)

## 2. Row counts and nominal totals by program × state × FY
Awards are counted by base action FY, and the dollars are each award's total obligation to date. The second table sums transaction obligations by action FY.

In [ ]:
t_awards = (A.groupby(["program_id", "geo_state", "fy"])
              .agg(awards=("assistance_award_unique_key", "nunique"), total_obligated_nominal=("total_obligated_nominal", "sum"))
              .reset_index())
pd.set_option("display.max_rows", 400)
t_awards.pivot_table(index=["program_id", "geo_state"], columns="fy", values="awards", fill_value=0, aggfunc="sum")

In [ ]:
(T.groupby(["program_id", "geo_state", "fy"])["federal_action_obligation_nominal"].sum()
   .unstack("fy").fillna(0).div(1e6).round(2))  # $M nominal, by action FY

In [ ]:
summary = (A.groupby(["program_id", "geo_state"])
             .agg(awards=("assistance_award_unique_key", "nunique"),
                  obligated_m=("total_obligated_nominal", lambda s: s.sum() / 1e6))
             .round(2))
summary

## 3. Recipient identifiers (UEI / legacy DUNS)
SAM.gov switched from DUNS to UEI in April 2022, so older awards may have only a DUNS.

In [ ]:
ids = A.assign(uei_or_duns=A.has_uei | A.has_duns)
pd.concat([
    ids.groupby("program_id")[["has_uei", "has_duns", "uei_or_duns"]].mean().round(3),
    ids.groupby("program_id").size().rename("awards"),
], axis=1)

In [ ]:
ids.assign(era=pd.cut(ids.fy.astype(float), [2009, 2021, 2026], labels=["FY2010-21", "FY2022-26"])) \
   .groupby(["program_id", "era"], observed=True)[["has_uei", "has_duns", "uei_or_duns"]].mean().round(3)

In [ ]:
sub_ids = S[S.in_coastal_county]
sub_ids.groupby("program_id")[["has_uei", "has_duns"]].mean().round(3).assign(subawards=sub_ids.groupby("program_id").size())

## 4. BRIC: state-held prime dollars and what subawards identify

In [ ]:
B = A[A.aln == "97.047"]
bric_state = B.groupby("recipient_is_state_gov")["total_obligated_nominal"].agg(["count", "sum"])
bric_state["share_of_dollars"] = (bric_state["sum"] / bric_state["sum"].sum()).round(3)
bric_state

In [ ]:
state_primes = B[B.recipient_is_state_gov]
sb = subs[(subs.aln == "97.047") & subs.prime_award_unique_key.isin(state_primes.assistance_award_unique_key)]
rows = []
for st, g in state_primes.groupby("recipient_state"):
    s_all = sb[sb.prime_award_unique_key.isin(g.assistance_award_unique_key)]
    s_coast = s_all[s_all.in_coastal_county]
    prime_d = g.total_obligated_nominal.sum()
    rows.append({"state": st, "state_primes": len(g), "prime_obligated_m": prime_d / 1e6,
                 "subawards": len(s_all), "subaward_m": s_all.subaward_amount_nominal.sum() / 1e6,
                 "share_identified_by_subawards": s_all.subaward_amount_nominal.sum() / prime_d if prime_d else None,
                 "coastal_subawards": len(s_coast), "coastal_subaward_m": s_coast.subaward_amount_nominal.sum() / 1e6})
pd.DataFrame(rows).round(3)

Caveat: subaward amounts are FFATA-reported and can be double-reported or amended, so the identified share may go above 1. The prime total is the obligation to date.

In [ ]:
sub_geo = S[S.aln == "97.047"]
pd.Series({"subawards_in_states": len(sub_geo), "zip_unmatched": sub_geo.zip_unmatched.sum(),
           "zip_multi_county": sub_geo.zip_multi_county.sum(), "in_coastal_county": sub_geo.in_coastal_county.sum()})

## 5. Earmark flags
These are text-based flags for the pilot. The definitive earmark tags come from the appropriations CPF/CDS tables (T2.4). The funding-opportunity table below shows which FONs carry the earmarks, so the rule can be tightened after the first run.

In [ ]:
E = A[A.aln.isin(["20.823", "97.047"])]
(E.groupby(["aln", "program_id", "program_basis"])
   .agg(awards=("assistance_award_unique_key", "nunique"), obligated_m=("total_obligated_nominal", lambda s: s.sum() / 1e6),
        earmark_flagged=("earmark_flag", "sum"))
   .round(2))

In [ ]:
E.assign(fon=E.funding_opportunity_number.fillna("(blank)")) \
 .pivot_table(index=["aln", "fon"], columns="fy", values="assistance_award_unique_key", aggfunc="nunique", fill_value=0)

In [ ]:
E[E.earmark_flag][["aln", "program_id", "fy", "recipient_name", "geo_state", "total_obligated_nominal",
                   "funding_opportunity_number", "prime_award_base_transaction_description"]].sort_values(["aln", "fy"])

## 6. Harbor keyword flag and geography match basis
`harbor_kw_flag` applies only the inventory's keyword rule. The crosswalk part of the rule comes later (T1.2/T1.3). PIDP is `all` (a port program).

In [ ]:
pd.concat([A.groupby("program_id")["harbor_kw_flag"].agg(["sum", "mean"]).rename(columns={"sum": "flagged", "mean": "share"}),
           A[A.harbor_kw_flag].groupby("program_id")["total_obligated_nominal"].sum().div(1e6).rename("flagged_m")], axis=1).round(3)

In [ ]:
pd.crosstab(A.program_id, A.geo_match_basis.replace("", "(state-level BRIC prime)"), margins=True)

In [ ]:
if "eda_rlf_flag" in A:
    print(A[A.aln.isin(["11.300", "11.307"])].groupby("program_id")["eda_rlf_flag"].sum())

## 7. Independent count checks
One program-year per program, compared with a published figure. Each external number and its source are filled in from a document I read, and the `source` string says which one. A difference is explained in the handoff, not forced to zero.

In [ ]:
EXTERNAL_CHECKS = [
    # {"program_id": ..., "fy": ..., "states": [...], "external_count": ..., "source": "..."}
]

def our_count(pid, fy, states):
    g = awards[(awards.program_id.str.startswith(pid)) & (awards.fy == fy) & awards.in_states &
               (awards.recipient_state.isin(states) | awards.pop_state.isin(states))]
    return g.assistance_award_unique_key.nunique(), g.total_obligated_nominal.sum()

pd.DataFrame([{**c, "usaspending_count": our_count(c["program_id"], c["fy"], c["states"])[0],
               "usaspending_obligated": our_count(c["program_id"], c["fy"], c["states"])[1]}
              for c in EXTERNAL_CHECKS])